In [26]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [27]:
class BMIInput(BaseModel):
    weight_kg: float = Field(
        gt=0,
        description="Body weight in kilograms, greater than zero"
    )
    height_cm: float = Field(
        gt=0,
        description="Height in centimeters, greater than zero"
    )


print("BMI input schema created.")

BMI input schema created.


In [28]:
class CalorieInput(BaseModel):
    weight_kg: float = Field(
        gt=0,
        description="Body weight in kilograms"
    )
    height_cm: float = Field(
        gt=0,
        description="Height in centimeters"
    )
    age: int = Field(
        gt=0,
        description="Age in years"
    )
    activity_level: str = Field(
        description=(
            "Activity level: sedentary, lightly_active, "
            "moderately_active, or very_active"
        )
    )


print("Calorie needs input schema created.")

Calorie needs input schema created.


In [29]:
def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """Calculate BMI using weight in kilograms and height in centimeters."""

    if weight_kg <= 0 or height_cm <= 0:
        raise ValueError("Weight and height must be positive.")

    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    return (
        f"Weight: {weight_kg:.2f} kg\n"
        f"Height: {height_cm:.2f} cm\n"
        f"BMI: {bmi:.2f}"
    )


bmi_tool = StructuredTool.from_function(
    func=calculate_bmi,
    name="calculate_bmi",
    description=(
        "Calculate body mass index (BMI) using weight in kilograms "
        "and height in centimeters."
    ),
    args_schema=BMIInput
)

print("BMI StructuredTool created.")

BMI StructuredTool created.


In [30]:
def calculate_daily_calories(
    weight_kg: float,
    height_cm: float,
    age: int,
    activity_level: str
) -> str:
    """Estimate daily maintenance calories from weight, height, age, and activity."""

    if weight_kg <= 0 or height_cm <= 0 or age <= 0:
        raise ValueError("Weight, height, and age must be positive.")

    activity_multipliers = {
        "sedentary": 1.2,
        "lightly_active": 1.375,
        "moderately_active": 1.55,
        "very_active": 1.725
    }

    activity_level = activity_level.strip().lower()

    if activity_level not in activity_multipliers:
        raise ValueError(
            "Invalid activity level. Choose: sedentary, lightly_active, "
            "moderately_active, or very_active."
        )

    # Mifflin–St Jeor equation using the +5 constant
    bmr = (
        10 * weight_kg
        + 6.25 * height_cm
        - 5 * age
        + 5
    )

    estimated_calories = bmr * activity_multipliers[activity_level]

    return (
        f"Weight: {weight_kg:.2f} kg\n"
        f"Height: {height_cm:.2f} cm\n"
        f"Age: {age} years\n"
        f"Activity level: {activity_level}\n"
        f"Estimated daily maintenance calories: "
        f"{estimated_calories:.0f} kcal/day"
    )


calorie_tool = StructuredTool.from_function(
    func=calculate_daily_calories,
    name="calculate_daily_calories",
    description=(
        "Estimate daily maintenance calories using weight in kilograms, "
        "height in centimeters, age in years, and activity level."
    ),
    args_schema=CalorieInput
)

print("Daily calorie StructuredTool created.")

Daily calorie StructuredTool created.


In [31]:
class HealthToolkit:
    def __init__(self):
        self.bmi_tool = bmi_tool
        self.calorie_tool = calorie_tool

    def get_tools(self):
        return [
            self.bmi_tool,
            self.calorie_tool
        ]


health_toolkit = HealthToolkit()
tools = health_toolkit.get_tools()

print("HealthToolkit created.")
print("Available tools:")

for item in tools:
    print("-", item.name)

HealthToolkit created.
Available tools:
- calculate_bmi
- calculate_daily_calories


In [32]:
sample_inputs = {
    "weight_kg": 70,
    "height_cm": 175,
    "age": 25,
    "activity_level": "moderately_active"
}

manual_bmi = bmi_tool.invoke({
    "weight_kg": sample_inputs["weight_kg"],
    "height_cm": sample_inputs["height_cm"]
})

manual_calories = calorie_tool.invoke(sample_inputs)

print("Manual BMI result:")
print(manual_bmi)

print("\nManual daily calorie result:")
print(manual_calories)

Manual BMI result:
Weight: 70.00 kg
Height: 175.00 cm
BMI: 22.86

Manual daily calorie result:
Weight: 70.00 kg
Height: 175.00 cm
Age: 25 years
Activity level: moderately_active
Estimated daily maintenance calories: 2594 kcal/day


In [33]:
user_query = """
For a person weighing 70 kg, height 175 cm, age 25,
and moderately active, calculate both BMI and estimated daily
maintenance calories.

You MUST call both tools in this response:
1. calculate_bmi with weight_kg=70 and height_cm=175.
2. calculate_daily_calories with weight_kg=70, height_cm=175,
   age=25, and activity_level="moderately_active".

Do not answer until you have requested both tools.
"""

messages = [HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages)

print("Model-suggested tool calls:")
for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

Model-suggested tool calls:
Tool: calculate_bmi
Arguments: {'height_cm': 175, 'weight_kg': 70}


In [34]:
messages.append(response)

tool_map = {
    item.name: item for item in tools
}

tool_results = []
executed_tools = set()

# Execute the model-suggested calls first
for call in response.tool_calls:
    tool_name = call["name"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool: {tool_name}")

    result = tool_map[tool_name].invoke(call["args"])
    tool_results.append((tool_name, result))
    executed_tools.add(tool_name)

    print(f"\nModel-suggested tool executed: {tool_name}")
    print(result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

# Execute the missing tool manually if the model omitted it
if "calculate_daily_calories" not in executed_tools:
    calorie_args = {
        "weight_kg": 70,
        "height_cm": 175,
        "age": 25,
        "activity_level": "moderately_active"
    }

    calorie_result = calorie_tool.invoke(calorie_args)

    tool_results.append((
        "calculate_daily_calories",
        calorie_result
    ))

    executed_tools.add("calculate_daily_calories")

    print("\nMissing tool executed manually: calculate_daily_calories")
    print(calorie_result)

print("\nTools executed:", executed_tools)


Model-suggested tool executed: calculate_bmi
Weight: 70.00 kg
Height: 175.00 cm
BMI: 22.86

Missing tool executed manually: calculate_daily_calories
Weight: 70.00 kg
Height: 175.00 cm
Age: 25 years
Activity level: moderately_active
Estimated daily maintenance calories: 2594 kcal/day

Tools executed: {'calculate_daily_calories', 'calculate_bmi'}


In [35]:
tool_results_text = "\n\n".join(
    f"{name}:\n{result}"
    for name, result in tool_results
)

final_response = llm.invoke([
    HumanMessage(
        content=(
            "Summarize both health calculations using the exact results "
            "below. Do not change the values. Explain that calorie needs "
            "are estimates, not medical advice.\n\n"
            f"{tool_results_text}"
        )
    )
])

print("Final health metrics result:")
print(final_response.content)

Final health metrics result:
**Health Calculations Summary**

| Calculation | Input Values | Result |
|-------------|--------------|--------|
| **BMI** | Weight: 70.00 kg <br>Height: 175.00 cm | **BMI: 22.86** |
| **Daily Calorie Needs** | Weight: 70.00 kg <br>Height: 175.00 cm <br>Age: 25 years <br>Activity level: moderately_active | **Estimated daily maintenance calories: 2594 kcal/day** |

*Note:* These calorie figures are **estimates** based on standard formulas and the activity level you provided. They are not a substitute for personalized medical or nutritional advice. Always consult a qualified health professional for guidance tailored to your individual needs.


In [36]:
required_tools = {
    "calculate_bmi",
    "calculate_daily_calories"
}

called_tools = {
    call["name"] for call in response.tool_calls
}

executed_tools = {
    name for name, result in tool_results
}

print("Model-suggested tools:", called_tools)
print("Executed tools:", executed_tools)

assert required_tools.issubset(executed_tools)
assert final_response.content.strip()

print("Validation passed: both tool results were generated.")
print(
    "Note: the model suggested only the tools listed above; "
    "any missing tool was executed manually."
)

Model-suggested tools: {'calculate_bmi'}
Executed tools: {'calculate_daily_calories', 'calculate_bmi'}
Validation passed: both tool results were generated.
Note: the model suggested only the tools listed above; any missing tool was executed manually.


Model-suggested tool calls:
Tool: calculate_bmi
Arguments: {'height_cm': 175, 'weight_kg': 70}
